# RxRx3 表征比较：CA-MAE、DiffMAE、CNN-MAE 与 MAE 参考

本 notebook 比较三种新增方法，并保留普通 MAE 作为结构参考。另读取项目已有的 CellProfiler、MAE-G、MAE-L、OpenPhenom 表征作外部参考。这些公开表征的预训练数据、模型大小和计算预算不同，不能把其成绩解释为本项目 50 轮训练的公平胜负。

默认只执行已有 embedding 的小样本评价，不启动训练、不覆盖 embedding。没有训练结果的模型会明确跳过。表格均由真实文件计算，不预填模型优劣结论。默认抽样用于验证评估流程；正式结论应使用完整的共同测试集、多随机种子和按扰动分组的置信区间。

## 1. 方法与固定实验条件

| 方法 | 编码与重建目标 | 表征提取 | 需要验证的优势与代价 |
|---|---|---|---|
| 普通 MAE（参考） | 多通道联合 patch；可见 token 编码，重建被遮挡像素 | 完整干净图像的 token 均值 | 结构简单；通道组合固定 |
| CA-MAE | 每个通道分别生成 patch token，共享编码器；掩码自编码 | 完整干净图像跨通道/token 均值 | 可处理通道数变化，但 token 数明显增加 |
| DiffMAE | 可见区域作为条件，对遮挡区域的带噪图像去噪 | 完整干净图像编码器均值 | 去噪目标可能提高表征；扩散时间步与重建评估更复杂 |
| CNN-MAE | 卷积编码器/解码器，对掩码区域做像素重建 | 全局平均池化后 512 维 | 局部归纳偏置；隐藏像素先清零，稠密卷积仍处理隐藏网格，与只处理可见 token 的 ViT 成本不同 |

输入沿用项目完整 **512×512、六通道**；patch=**8×8 像素**，mask ratio=**0.75**，训练 **50 epochs**。三个 Transformer 使用 encoder dim=512、decoder dim=256，二者均 8 heads；深度沿用 12/4。CNN 没有 attention heads。除模型结构与重建目标外使用相同数据、划分、预处理和优化配置。CNN 的 8×8 是掩码网格；不能声称其卷积感受野等于 ViT 的 patch。

普通 MAE 每图 4096 个完整 token；六通道 CA-MAE 为 24576 个，75% 掩码后仍有 6144 个可见 token。50 轮相同并不意味着耗时、显存或训练 FLOPs 相同，必须单独报告。CA 的 channel-concat 池化可作为补充实验，但六通道时为 3072 维，主比较用各模型 native 512 维均值/GAP。当前实现没有 CLS token，不做 CLS 对比。

MAE 重建损失为 $L=|M|^{-1}\sum_{i\in M}\|\hat{x}_i-x_i\|_2^2$。DiffMAE 训练/默认评价的 sampled-t 单步去噪损失与此难度不同，**不得直接按默认 reconstruction loss 排名**；如需比较修复效果，应从相同掩码噪声起点运行固定步数采样，再测相同区域误差。主评价始终使用干净完整图像的 encoder embedding。

In [ ]:
from pathlib import Path
import os, sys, json, itertools, subprocess, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src' / 'cellvit').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'cellvit').is_dir():
    raise RuntimeError('请从项目根目录或 notebooks 目录运行。')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from cellvit.data.source import load_metadata, make_plate_splits
from cellvit.embedding_metrics import (load_embedding_parquet, align_embedding_tables,
    retrieval_metrics, gene_retrieval_metrics, percent_replicating,
    evaluation_metadata, unit_vectors)

SEED = 2026
SOURCE_ROOT = Path(os.environ.get('RXRX3_CORE_ROOT', 'E:/CellPainting/rxrx3_core'))
MDS_ROOT = Path(os.environ.get('RXRX3_MDS_ROOT', os.environ.get('CELLVIT_MDS_ROOT', str(PROJECT_ROOT / 'data' / 'rxrx3_mds'))))
RUN_ROOT = Path(os.environ.get('CELLVIT_COMPARISON_ROOT',
    str(PROJECT_ROOT / 'runs' / 'comparison_patch8_50epochs')))
METADATA_CSV = SOURCE_ROOT / 'metadata_rxrx3_core.csv'
EVAL_SPLIT = 'val'  # 先在 val 确定方案；test 仅最终评价，勿按 test 结果挑模型。
RUN_TRAINING = False
EXPORT_EMBEDDINGS = False
RUN_DIFFUSION_RECONSTRUCTION = False
INCLUDE_PUBLIC_REFERENCES = True
SMOKE_SAMPLE = True
MAX_GENE_LABELS = 20
MAX_COMPOUND_LABELS = 80
MAX_VISUALIZATION_WELLS = 800
TRAIN_TRANSFORM_WELLS = 1024
N_BOOTSTRAP = 200 if SMOKE_SAMPLE else 1000
BLOCK_SIZE = 64
RUN_PR = True
RUN_UMAP = False
assert EVAL_SPLIT in {'val', 'test'}

MODEL_CONFIGS = {
    'MAE': ('mae', 'mae_comparison.json'),
    'CA-MAE': ('ca_mae', 'ca_mae.json'),
    'DiffMAE': ('diffmae', 'diffmae.json'),
    'CNN-MAE': ('cnn_mae', 'cnn_mae.json'),
}
PUBLIC_FILES = {
    'CellProfiler [external]': 'CellProfiler_features_rxrx3_core.parquet',
    'MAE-G [external]': 'MAE-G_8_PP-16M_trimmed_rxrx3_core_embeddings.parquet',
    'MAE-L [external]': 'MAE-L_8_RPI-93M_rxrx3_core_embeddings.parquet',
    'OpenPhenom [external]': 'OpenPhenom_rxrx3_core_embeddings.parquet',
}
print('Project:', PROJECT_ROOT)
print('Source:', SOURCE_ROOT, '| MDS:', MDS_ROOT)
print('Evaluation split:', EVAL_SPLIT, '| smoke sample:', SMOKE_SAMPLE)

## 2. 可选训练与导出

先检查下列打印的命令，再将对应开关改为 True。四个模型使用独立输出目录。训练需在安装 PyTorch 的项目环境执行；默认评价已有 parquet 不导入 PyTorch。新训练模型导出相同 split 的 well 级 embedding；为 train-only PCA 另导出 train split。遇到已有 embedding 文件直接跳过，避免覆盖。完整训练还可使用项目 `scripts/train_comparison.sh`。

MDS_ROOT 必须指向已有 MDS 数据集；路径仅示例，先按本机数据设置。单图包含大量 token，batch size=1、累积 64 次，实际显存仍取决于设备和注意力实现。出现显存不足应记录后统一比较训练预算，不能只对某一个方法静默更改输入大小。

In [ ]:
training_commands, export_commands = [], []
for label, (folder, config_file) in MODEL_CONFIGS.items():
    run_dir = RUN_ROOT / folder
    cmd = ['cellvit-train', '--mds-root', str(MDS_ROOT), '--output', str(run_dir),
           '--model-config', str(PROJECT_ROOT / 'configs' / config_file),
           '--epochs', '50', '--batch-size', '1', '--accumulation-steps', '64',
           '--seed', str(SEED)]
    training_commands.append(cmd)
    print(label, subprocess.list2cmdline(cmd))
    if RUN_TRAINING:
        subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
    for split in ('train', 'val', 'test'):
        target = run_dir / f'embedding_{split}.parquet'
        cmd = ['cellvit-embed', '--checkpoint', str(run_dir / 'best_reconstruction.pt'),
               '--mds-root', str(MDS_ROOT), '--split', split,
               '--output', str(target), '--batch-size', '1']
        export_commands.append(cmd)
        if EXPORT_EMBEDDINGS and not target.exists():
            subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

if RUN_DIFFUSION_RECONSTRUCTION:
    target = RUN_ROOT / 'diffmae' / 'reconstruction_ddim50.json'
    if target.exists():
        raise FileExistsError(f'请选择新的结果文件，避免覆盖：{target}')
    subprocess.run(['cellvit-evaluate', '--checkpoint',
        str(RUN_ROOT / 'diffmae' / 'best_reconstruction.pt'),
        '--mds-root', str(MDS_ROOT), '--split', 'val', '--max-batches', '4',
        '--diffusion-sampling-steps', '50', '--output', str(target)],
        cwd=PROJECT_ROOT, check=True)

## 3. 评价对象、划分与共同样本

数据中的多张图像先在 **well** 级聚合；同 well 的视野不能互作独立重复。使用完整 metadata 计算项目 plate split，之后才筛选 val/test 和抽样，避免抽样改变划分。所有参与主比较的表征按 well_id 连接 metadata，并取共同 well 集合；同一个相似度指标再次去除所有方法中任一无效的行（cosine 零向量、Pearson 常量向量）。这样每个模型看到相同查询和候选。

主协议沿用 Project.md：**基因**跨 experiment、跨 guide 检索相同基因；每个候选 experiment 独立构建基因原型。**化合物**在相同 experiment、相同 dose 内跨 plate 检索同一化合物。阴性对照不作为扰动标签正例。若某 split 没有足够跨 plate/guide 重复，报告不可评价和覆盖率，不放宽协议凑分数。

默认结构抽样优先保留跨 experiment 的共享基因，以及单个 experiment-dose 下的多 plate 化合物；它服务于运行检查，具有选择偏倚。关闭 SMOKE_SAMPLE 才评价完整共同 split。没有 MoA、通路或毒性真值时不计算相应生物关系指标。

In [ ]:
metadata = load_metadata(METADATA_CSV)
plate_splits = make_plate_splits(metadata, seed=SEED)  # 必须在全量 metadata 上计算。
def split_of(row):
    return plate_splits[(str(row['experiment_name']), int(row['plate']))]
all_rows = [dict(row, well_id=well_id) for well_id, row in metadata.items()]
split_rows = [row for row in all_rows if split_of(row) == EVAL_SPLIT]
train_rows = [row for row in all_rows if split_of(row) == 'train']

def nonempty(value):
    return str(value).strip() not in {'', 'nan', 'None', 'DMSO', 'EMPTY', 'UNTREATED'}

def structural_sample(rows):
    if not SMOKE_SAMPLE:
        return sorted(row['well_id'] for row in rows)
    frame = pd.DataFrame(evaluation_metadata(rows)).fillna('')
    gene = frame[frame['is_query'] & frame['perturbation_type'].eq('CRISPR')]
    chosen = set()
    if not gene.empty:
        by_exp = {str(exp): set(g['gene']) for exp, g in gene.groupby('experiment_name')}
        pairs = list(itertools.combinations(sorted(by_exp), 2))
        if pairs:
            exp_a, exp_b = max(pairs, key=lambda p: len(by_exp[p[0]] & by_exp[p[1]]))
            labels = sorted(by_exp[exp_a] & by_exp[exp_b])[:MAX_GENE_LABELS]
            keep = gene[gene['experiment_name'].isin([exp_a, exp_b]) & gene['gene'].isin(labels)]
            chosen.update(keep['well_id'])
    # 按真实 Query Compounds 注释识别化合物；保留一个 experiment-dose 的完整跨 plate 重复。
    compound = frame[frame['is_query'] & frame['perturbation_type'].eq('COMPOUND')]
    if not compound.empty:
        blocks = list(compound.groupby(['experiment_name', 'concentration'], dropna=False))
        eligible = [(key, g) for key, g in blocks if g['plate'].nunique() >= 2]
        if eligible:
            _, block = max(eligible, key=lambda item: item[1]['treatment'].nunique())
            labels = sorted(block['treatment'].unique())[:MAX_COMPOUND_LABELS]
            chosen.update(block[block['treatment'].isin(labels)]['well_id'])
    # 仍可能缺少满足协议的重复；保留确定性小样本以输出不可评价原因。
    if not chosen:
        chosen.update(sorted(frame['well_id'])[:MAX_VISUALIZATION_WELLS])
    return sorted(chosen)

eval_well_ids = structural_sample(split_rows)
rng = np.random.default_rng(SEED)
train_ids = sorted(row['well_id'] for row in train_rows)
train_fit_ids = sorted(rng.choice(train_ids, min(TRAIN_TRANSFORM_WELLS, len(train_ids)), replace=False).tolist())
print(f'metadata wells={len(metadata):,}; {EVAL_SPLIT} wells={len(split_rows):,}; selected={len(eval_well_ids):,}')
print('train-only transform reference wells:', len(train_fit_ids))
if not eval_well_ids:
    raise RuntimeError('选定 split 没有 well，请检查 metadata 和划分。')

In [ ]:
tables, train_tables, missing = {}, {}, []
for label, (folder, _) in MODEL_CONFIGS.items():
    path = RUN_ROOT / folder / f'embedding_{EVAL_SPLIT}.parquet'
    train_path = RUN_ROOT / folder / 'embedding_train.parquet'
    if path.exists():
        tables[label] = load_embedding_parquet(path, well_ids=eval_well_ids)
        if train_path.exists():
            train_tables[label] = load_embedding_parquet(train_path, well_ids=train_fit_ids)
    else:
        missing.append((label, str(path)))
if INCLUDE_PUBLIC_REFERENCES:
    # 只保留请求的行；分批扫描 parquet，不将 1–2 GB 文件整体载入内存。
    requested = sorted(set(eval_well_ids) | set(train_fit_ids))
    for label, filename in PUBLIC_FILES.items():
        path = SOURCE_ROOT / filename
        if path.exists():
            loaded = load_embedding_parquet(path, well_ids=requested)
            tables[label] = loaded
            train_tables[label] = loaded
        else:
            missing.append((label, str(path)))
for label, path in missing:
    print('未提供，跳过:', label, path)
if not tables:
    raise RuntimeError('没有可用 embedding，请先设置 SOURCE_ROOT，或启用导出。')
values, rows, alignment_report = align_embedding_tables(
    tables, metadata, well_ids=eval_well_ids, drop_nonfinite=True)
display(pd.DataFrame([{'representation': name, 'n_wells': len(x), 'dimension': x.shape[1]}
                      for name, x in values.items()]))
display(alignment_report)
if not rows:
    raise RuntimeError('所有表征没有共同有效 well；检查 well_id、split 和文件。')

# 不同距离分别取共同有效行，防止某模型单独删除零向量造成不同候选集。
metric_data = {}
for metric in ('cosine', 'pearson'):
    valid = np.ones(len(rows), dtype=bool)
    for x in values.values():
        valid &= unit_vectors(x, metric=metric)[1]
    metric_data[metric] = ({name: x[valid] for name, x in values.items()},
                           [row for row, keep in zip(rows, valid) if keep])
    print(metric, ': removed across all models =', int((~valid).sum()), '; retained =', int(valid.sum()))

## 4. 相似度与两类主检索指标

**Cosine**：$s(x,y)=x^Ty/(\|x\|_2\|y\|_2)$，比较方向而弱化向量模长。**Pearson**：先分别对每个向量减去其维度均值，再计算 cosine；它弱化每个样本所有维度共同的加性偏移，但不等于按特征进行 train-set 标准化。不同模型的相似度绝对值尺度不可直接当质量分数，应看正负例分离和检索表现。

给定查询 q，相关候选数为 $R_q$，排序第 k 位的相关性为 $r_k$：$AP_q=R_q^{-1}\sum_k P_q(k)r_k$；mAP 是各扰动组的宏平均 AP，防止重复数多的扰动主导。$Recall@K=\sum_{k\leq K}r_k/R_q$；**Hit@K** 为前 K 是否至少一个正例，两者不同。$RR_q=1/\operatorname{rank}(first positive)$，MRR 对第一个相关结果敏感，适合基因原型检索。

主报告：基因 MRR、Recall@1/5；化合物 mAP、Recall@1/5。保留 random AP/Recall、可评价 query 数、覆盖率和排除原因。置信区间按扰动组 bootstrap，不把同一扰动的多个 well 当作独立样本。随机基线取决于候选数与正例数，不能简单用一个固定常数。

In [ ]:
retrieval_results, score_rows = {}, []
for metric, (matrices, metric_rows) in metric_data.items():
    for name, x in matrices.items():
        for protocol, fn in [('gene', gene_retrieval_metrics), ('compound', retrieval_metrics)]:
            result = fn(x, metric_rows, metric=metric, ks=(1, 5, 10),
                        block_size=BLOCK_SIZE, seed=SEED, n_bootstrap=N_BOOTSTRAP)
            retrieval_results[(name, metric, protocol)] = result
            summary = result['summary']
            record = {'representation': name, 'similarity': metric, 'protocol': protocol}
            for key in ('mrr', 'map', 'recall@1', 'recall@5', 'hit@1', 'random_ap',
                        'random_recall@1', 'coverage', 'n_evaluable_queries', 'n_groups'):
                record[key] = summary.get(key, np.nan)
            record['ci95'] = summary.get('ci95', {})
            record['excluded_reasons'] = summary.get('excluded_reasons', {})
            score_rows.append(record)
scores = pd.DataFrame(score_rows)
display(scores)
print('只有 n_evaluable_queries > 0 的结果可解释；SMOKE_SAMPLE=True 时不推断全量优劣。')

## 5. 重复一致性：明确命名的 percent replicating 变体

对每个扰动组，计算满足跨 plate（基因还要求跨 guide）条件的组内成对相似度中位数 $T_g$。以匹配重复数量、plate/扰动类型/dose 结构的不同扰动组形成随机 null 分布；若 $T_g$ 超过其 95% 分位数，则该组被判为可重复。$PR=100\times\#replicating/\#evaluable$。必须同时报告可评价组数、抽样组数、null 次数与覆盖率；无有效匹配 null 的组不能直接计为失败。

[Way 等（Cell Systems，2022）原方法](https://pmc.ncbi.nlm.nih.gov/articles/PMC10246468/)使用 median pairwise **Spearman**，并按重复数、well position、dose 匹配 null。这里采用 **Pearson/cosine、plate/type/dose 匹配并增加跨 plate/guide 条件**，是本项目的严格命名变体，不是原论文的原样复现。默认少量组和 null 只用于运行检查；正式运行至少提高 null 到 1000、扩大组覆盖并检查置信区间稳定性。

In [ ]:
pr_results, pr_records = {}, []
if RUN_PR:
    for metric, (matrices, metric_rows) in metric_data.items():
        for name, x in matrices.items():
            result = percent_replicating(x, metric_rows, metric=metric, seed=SEED,
                n_null=100 if SMOKE_SAMPLE else 1000, n_bootstrap=N_BOOTSTRAP,
                max_groups=20 if SMOKE_SAMPLE else None, max_replicates=6 if SMOKE_SAMPLE else 20)
            pr_results[(name, metric)] = result
            s = result['summary']
            pr_records.append({'representation': name, 'similarity': metric,
                **{key: s.get(key) for key in ('percent_replicating', 'ci95',
                    'n_groups_available', 'n_groups_sampled', 'n_evaluable_groups', 'coverage')}})
    display(pd.DataFrame(pr_records))

## 6. 正负例相似度分布

下图仅用于诊断。正例为同一基因的跨 experiment、跨 guide 样本，或同 experiment、同 dose 的跨 plate 同化合物样本；负例来自相同协议下不同标签。每种相似度使用所有模型的共同有效样本和相同成对索引。观察分布重叠、尾部和中位差；不能只比较相似度是否接近 1。负例与正例比例由抽样决定，图形不替代 mAP/MRR。

In [ ]:
def pair_label(row):
    if not row['is_query']:
        return None
    kind = 'gene' if row['perturbation_type'] == 'CRISPR' else 'compound'
    return kind, row['perturbation_label']

def comparable_pair(a, b):
    la, lb = pair_label(a), pair_label(b)
    if la is None or lb is None or la[0] != lb[0]:
        return False
    if la[0] == 'gene':
        return a['experiment_name'] != b['experiment_name'] and a['guide'] != b['guide']
    return (a['experiment_name'] == b['experiment_name']
            and a['dose'] == b['dose'] and a['batch_id'] != b['batch_id'])

for metric, (matrices, rr) in metric_data.items():
    rr = evaluation_metadata(rr)
    rng_pairs = np.random.default_rng(SEED)
    pos, neg = [], []
    # 随机抽取的候选成对索引固定；避免构造全量 N×N 相似度矩阵。
    for _ in range(min(100000, max(1, len(rr)) * 100)):
        if len(rr) < 2:
            break
        i, j = rng_pairs.integers(len(rr), size=2)
        if i == j or not comparable_pair(rr[i], rr[j]):
            continue
        target = pos if pair_label(rr[i]) == pair_label(rr[j]) else neg
        if len(target) < 5000:
            target.append((int(i), int(j)))
    fig, axes = plt.subplots(1, len(matrices), figsize=(5 * len(matrices), 3), squeeze=False)
    for ax, (name, x) in zip(axes.flat, matrices.items()):
        unit, _ = unit_vectors(x, metric=metric)
        for label, pairs in [('positive', pos), ('negative', neg)]:
            if pairs:
                pairs = np.asarray(pairs)
                similarity = np.sum(unit[pairs[:, 0]] * unit[pairs[:, 1]], axis=1)
                ax.hist(similarity, bins=35, density=True, alpha=.5, label=f'{label} n={len(pairs)}')
        ax.set_title(name); ax.set_xlabel(metric); ax.set_ylabel('density')
        if pos or neg:
            ax.legend()
    plt.tight_layout(); plt.show()
    print(metric, 'sampled positive/negative pairs:', len(pos), len(neg))

## 7. native embedding、train-only 标准化/PCA 与批次审计

主表使用各方法原生表征（新模型均 512 维；公开参考可能不同）。维数不同也会改变相似度分布。补充表采用训练 split 拟合每维均值/标准差，再在训练 split 拟合相同维数 PCA，并变换 val/test；PCA 降维保留训练数据方差较大的线性方向，不保证保留生物信号。test 绝不拟合 scaler/PCA，也不据 test 挑选维数。

CellProfiler 各特征量纲差异大，原始 cosine/Pearson 可能被高方差特征主导，需重点对照 train-only 标准化/PCA 后结果。训练参考样本同样取共同 train well；若某模型缺少 train embedding，跳过其变换，不能用测试数据补齐。

图中 PCA 拟合于 train，着色使用 experiment+plate。批次 silhouette 是当前样本的描述统计，不是排除生物混杂后的技术批次强度估计；它直接在原生表征上计算：$s=(b-a)/\max(a,b)$，a 为同批次平均距离，b 为最近其他批次平均距离。高值说明批次聚集，但细胞类型、处理分布与批次可能混杂；低值也可能代表生物信号被抹去。因此须与跨批次生物检索联合分析。UMAP 仅辅助可视化，不能以二维簇更漂亮判断优劣。

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

transformed, pca_models, train_alignment = {}, {}, None
if train_tables:
    train_values, transform_rows, train_alignment = align_embedding_tables(
        train_tables, metadata, well_ids=train_fit_ids, drop_nonfinite=True)
    display(train_alignment)
    if len(transform_rows) >= 3:
        # 维数在 val 决定；这里固定 64，不根据 test 成绩选择。
        common_dim = min(64, len(transform_rows) - 1, *(x.shape[1] for x in train_values.values()))
        for name, x_train in train_values.items():
            if name not in values:
                continue
            scaler = StandardScaler().fit(x_train)
            pca = PCA(n_components=common_dim, svd_solver='randomized', random_state=SEED)
            pca.fit(scaler.transform(x_train))
            transformed[name] = pca.transform(scaler.transform(values[name]))
            pca_models[name] = (scaler, pca)
        print('train-fitted common PCA dimension:', common_dim)
    else:
        print('共同 train embedding 不足，跳过标准化/PCA。')
else:
    print('未提供 train embedding，跳过 train-only 标准化/PCA。')

pca_score_records = []
if transformed:
    # 与原生空间比较时，再取共同表示和共同有效井，不能比较不同候选集。
    shared_names = sorted(transformed)
    valid = np.ones(len(rows), dtype=bool)
    for name in shared_names:
        valid &= unit_vectors(transformed[name], metric='cosine')[1]
        valid &= unit_vectors(values[name], metric='cosine')[1]
    rr = [row for row, keep in zip(rows, valid) if keep]
    for name in shared_names:
        for representation, matrix in [('native', values[name][valid]), ('train-scaled-PCA', transformed[name][valid])]:
            for protocol, fn in [('gene', gene_retrieval_metrics), ('compound', retrieval_metrics)]:
                result = fn(matrix, rr, metric='cosine', ks=(1, 5), block_size=BLOCK_SIZE,
                            seed=SEED, n_bootstrap=N_BOOTSTRAP)
                s = result['summary']
                pca_score_records.append({'representation': name, 'space': representation,
                    'protocol': protocol, 'n_wells': len(rr), **{key: s.get(key) for key in
                    ('mrr', 'map', 'recall@1', 'recall@5', 'coverage')}})
    display(pd.DataFrame(pca_score_records))

In [ ]:
rng_vis = np.random.default_rng(SEED)
visual_indices = np.sort(rng_vis.choice(len(rows), min(MAX_VISUALIZATION_WELLS, len(rows)), replace=False))
visual_rows = [rows[i] for i in visual_indices]
batch_labels = np.asarray([f"{r['experiment_name']}/plate-{r['plate']}" for r in visual_rows])
label_codes, label_names = pd.factorize(batch_labels)
batch_records = []
for name, x in values.items():
    xx = x[visual_indices]
    valid = unit_vectors(xx, metric='cosine')[1]
    labels = batch_labels[valid]
    score = np.nan
    if 1 < len(np.unique(labels)) < len(labels):
        try:
            score = silhouette_score(xx[valid], labels, metric='cosine')
        except ValueError as error:
            print(name, 'batch silhouette unavailable:', error)
    batch_records.append({'representation': name, 'batch_cosine_silhouette': score,
                          'n_wells': int(valid.sum()), 'n_batches': len(np.unique(labels))})
display(pd.DataFrame(batch_records))

for name, coords in transformed.items():
    if coords.shape[1] < 2:
        continue
    fig, ax = plt.subplots(figsize=(6, 4))
    scatter = ax.scatter(coords[visual_indices, 0], coords[visual_indices, 1],
                         c=label_codes, s=10, alpha=.65, cmap='tab20')
    ax.set(title=name + ' | PCA fitted on train; colored by batch', xlabel='PC1', ylabel='PC2')
    if len(label_names) <= 12:
        handles, _ = scatter.legend_elements()
        ax.legend(handles, list(label_names)[:len(handles)], fontsize=7, loc='best')
    plt.tight_layout(); plt.show()

可选 UMAP 仍仅在训练参考数据上拟合，再 transform 测试点；公开训练参考样本有限时，局部结构也不稳定。默认不开启，缺包时跳过，不自动安装。若需要，可在项目环境手动运行 `python -m pip install 'umap-learn>=0.5,<0.6'`。UMAP 的随机种子、n_neighbors、min_dist 必须在 val 固定，不能反复调整到 test 看起来最好。

In [ ]:
if RUN_UMAP:
    try:
        import umap
    except ImportError:
        print('未安装 umap-learn；跳过可选图，核心评估不受影响。')
    else:
        for name, (scaler, pca) in pca_models.items():
            train_pca = pca.transform(scaler.transform(train_values[name]))
            if len(train_pca) < 4:
                continue
            reducer = umap.UMAP(n_components=2, n_neighbors=min(15, len(train_pca)-1),
                                min_dist=.1, random_state=SEED, transform_seed=SEED)
            reducer.fit(train_pca)
            coords = reducer.transform(transformed[name][visual_indices])
            fig, ax = plt.subplots(figsize=(6, 4))
            ax.scatter(coords[:, 0], coords[:, 1], c=label_codes, s=10, alpha=.65, cmap='tab20')
            ax.set_title(name + ' | train-fitted UMAP; batch colors')
            plt.tight_layout(); plt.show()

## 8. 效率、优势与局限的记录

记录参数量、实际完成 epochs、训练耗时、GPU 峰值显存、well/s 和 checkpoint 来源；没有实测就留空。推理速度应在同一设备、相同 batch、预热与计时范围下测量，区分图像读取、模型前向和 well 聚合。不能把参数量或默认 batch 当成显存实测。四种新增/参考实验的配置均来自 configs，实际训练结果以 run 目录记录为准。

In [ ]:
# 完成训练后填写实测值，例如 {'CA-MAE': {'train_hours': ..., 'peak_gpu_gb': ...}}。
# 不在 notebook 默认路径实例化大模型，也不为获取参数量导入 torch。
RESOURCE_OBSERVATIONS = {}
resource_records = []
for name, (folder, config_file) in MODEL_CONFIGS.items():
    config_path = PROJECT_ROOT / 'configs' / config_file
    config = json.loads(config_path.read_text(encoding='utf-8')) if config_path.exists() else {}
    resource_records.append({'representation': name, 'config': str(config_path),
        'target_epochs': 50, 'dimension_native': 512,
        'checkpoint_exists': (RUN_ROOT / folder / 'best_reconstruction.pt').exists(),
        'embedding_exists': (RUN_ROOT / folder / f'embedding_{EVAL_SPLIT}.parquet').exists(),
        'model_type': config.get('model_type', 'mae'),
        'train_hours': np.nan, 'peak_gpu_gb': np.nan, 'well_per_second': np.nan,
        'parameters': np.nan, 'completed_epochs': np.nan, **RESOURCE_OBSERVATIONS.get(name, {})})
display(pd.DataFrame(resource_records))

# 独立结果文件，不覆盖已有文件；只有主动改为 True 才写入。
SAVE_RESULTS = False
if SAVE_RESULTS:
    output_dir = RUN_ROOT / f'evaluation_{EVAL_SPLIT}_seed{SEED}'
    output_dir.mkdir(parents=True, exist_ok=True)
    target = output_dir / 'retrieval_scores.csv'
    if target.exists():
        raise FileExistsError(target)
    scores.to_csv(target, index=False)
    print('saved:', target)

## 9. 如何得出有根据的结论

1. **先查可比性**：相同 well/候选集合、是否完成相同训练轮次、覆盖率、null 匹配数和数据划分。公开预训练参考单列，不与本地 50 轮宣称同预算胜负。结构小样本结果仅说明流程可运行。
2. **先看生物检索**：基因看跨 experiment/guide 的 MRR 与 Recall@1/5，化合物看同 experiment/dose 跨 plate 的 mAP 与 Recall@K。两项任务可能结论不同，应分别报告。
3. **结合一致性与批次**：PR 高但跨板检索弱、批次 silhouette 高，可能说明实验结构而非生物效应主导；需检查剂量、细胞类型与标签分布的混杂。PR 低也可能由弱扰动和严格重复条件造成。
4. **检查表征处理**：native 与 train-scaled/PCA 的共同样本对比可揭示量纲、冗余维度或噪声影响。PCA 提升只说明该处理有用，不证明模型学到了更强机制。图像重建好也不必然意味着 embedding 好。
5. **量化不确定性和成本**：报告组级置信区间、至少多个训练种子、耗时和显存；接近或重叠的区间不足以支持稳定排名。DiffMAE 的 sampled-t 去噪与其他模型像素修复不是相同任务。

预期假设而非实验结论：CA-MAE 可能更适合通道变化，但 token 成本高；DiffMAE 的条件去噪可能促进语义结构，也增加训练/重建复杂度；CNN-MAE 可能更快且适合局部纹理，但其卷积掩码机制与 token 丢弃不同。最终优缺点必须由上述实际结果与资源测量决定。

## 参考资料

- [CA-MAE：Masked Autoencoders for Microscopy are Scalable Learners of Cellular Biology（Recursion）](https://arxiv.org/abs/2404.10242)；本项目采用六通道 token 化并共享通道处理的实现。
- [DiffMAE: Diffusion Models as Masked Autoencoders](https://arxiv.org/abs/2304.03283)。
- [Masked Autoencoders Are Scalable Vision Learners](https://arxiv.org/abs/2111.06377)。
- [Deep Residual Learning for Image Recognition](https://arxiv.org/abs/1512.03385)：CNN 残差结构的参考；本项目 CNN-MAE 是同数据自监督卷积基线，不是 ImageNet 预训练监督模型。
- [Percent replicating 与 Cell Painting/表达表征评价，Way 等](https://pmc.ncbi.nlm.nih.gov/articles/PMC10246468/)；本 notebook 已明确标注与原始 Spearman/null 匹配协议的差异。
- [Recursion RxRx3-core 基准与 EFAAR](https://github.com/recursionpharma/EFAAR_benchmarking/tree/trunk/RxRx3-core_benchmarks)：用于理解检索、重复与生物关系评价；当前没有外部 truth 的任务不计算。
- [copairs mAP 文档](https://cytomining.github.io/copairs/api/map/)：相似度排序评价的参考；当前实现的 candidate mask 和宏平均以本项目 `embedding_metrics.py`/Project.md 为准。

可复现记录：保留配置文件、seed、split、checkpoint、公共特征来源、共同 well 数、原生维数、预处理拟合范围、各协议覆盖率与硬件。不要将外部参考模型的预训练信息混入本项目的训练预算说明。